In [ ]:
! pip install cloudscraper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 2.3 MB/s eta 0:00:00


In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import time

# Định nghĩa cột cho DataFrame
columns = ['ID', 'Name', 'Age', 'Positions', 'Nationality', 'Overall', 'Potential', 'Club', 'Height', 'Weight', 'Value', 'Wage', 'Total attacking', 'Crossing', 'Finishing', 'Heading accuracy', 'Short passing', 'Volleys', 'Total skill', 'Dribbling', 'Curve', 'FK Accuracy', 'Long passing', 'Ball control', 'Total movement', 'Acceleration', 'Sprint speed', 'Agility', 'Reactions', 'Balance', 'Total power', 'Shot power', 'Jumping', 'Stamina', 'Strength', 'Long shots', 'Total mentality', 'Aggression', 'Interceptions', 'Att. Position', 'Vision', 'Penalties', 'Composure', 'Total defending', 'Defensive awareness', 'Standing tackle', 'Sliding tackle', 'Total goalkeeping', 'GK Diving', 'GK Handling', 'GK Kicking', 'GK Positioning', 'GK Reflexes', 'Total stats', 'Pace / Diving', 'Shooting / Handling', 'Passing / Kicking', 'Dribbling / Reflexes', 'Defending / Pace', 'Physical / Positioning', 'PlayStyles']

# Khởi tạo DataFrame rỗng
data = pd.DataFrame(columns=columns)

# Hàm lấy dữ liệu từ một trang
def get_data(offset):
    base_url = "https://sofifa.com/players?showCol%5B0%5D=ae&showCol%5B1%5D=hi&showCol%5B2%5D=wi&showCol%5B3%5D=oa&showCol%5B4%5D=pt&showCol%5B5%5D=vl&showCol%5B6%5D=wg&showCol%5B7%5D=ta&showCol%5B8%5D=cr&showCol%5B9%5D=fi&showCol%5B10%5D=he&showCol%5B11%5D=sh&showCol%5B12%5D=vo&showCol%5B13%5D=ts&showCol%5B14%5D=dr&showCol%5B15%5D=cu&showCol%5B16%5D=fr&showCol%5B17%5D=lo&showCol%5B18%5D=bl&showCol%5B19%5D=to&showCol%5B20%5D=ac&showCol%5B21%5D=sp&showCol%5B22%5D=ag&showCol%5B23%5D=re&showCol%5B24%5D=ba&showCol%5B25%5D=tp&showCol%5B26%5D=so&showCol%5B27%5D=ju&showCol%5B28%5D=st&showCol%5B29%5D=sr&showCol%5B30%5D=ln&showCol%5B31%5D=te&showCol%5B32%5D=ar&showCol%5B33%5D=in&showCol%5B34%5D=po&showCol%5B35%5D=vi&showCol%5B36%5D=pe&showCol%5B37%5D=cm&showCol%5B38%5D=td&showCol%5B39%5D=ma&showCol%5B40%5D=sa&showCol%5B41%5D=sl&showCol%5B42%5D=tg&showCol%5B43%5D=gd&showCol%5B44%5D=gh&showCol%5B45%5D=gc&showCol%5B46%5D=gp&showCol%5B47%5D=gr&showCol%5B48%5D=tt&showCol%5B49%5D=pac&showCol%5B50%5D=sho&showCol%5B51%5D=pas&showCol%5B52%5D=dri&showCol%5B53%5D=def&showCol%5B54%5D=phy&showCol%5B55%5D=ps1&r=250035&set=true"
    url = f"{base_url}&offset={offset * 60}"  # Đảm bảo offset được thêm đúng cách
    scraper = cloudscraper.create_scraper(
        browser={'browser': 'chrome', 'platform': 'windows', 'mobile': False},
        delay=10
    )
    print(f"Đang truy cập URL: {url}")  # In URL để kiểm tra
    try:
        response = scraper.get(url)
        if response.status_code != 200:
            print(f"Lỗi: Nhận mã trạng thái {response.status_code} cho offset {offset}")
            return pd.DataFrame(columns=columns)

        soup = BeautifulSoup(response.content, "html.parser")
        table_body = soup.find('tbody')
        if not table_body:
            print(f"Không tìm thấy tbody cho offset {offset}")
            return pd.DataFrame(columns=columns)

        rv = []
        for row in table_body.find_all('tr'):
            try:
                td = row.find_all('td')
                player_data = {}
                player_data['ID'] = td[0].find('img').get('id') if td[0].find('img') else "N/A"
                player_data['Nationality'] = td[1].find('img').get('title') if td[1].find('img') else "N/A"
                player_data['Name'] = td[1].find("a").get("data-tippy-content") if td[1].find("a") else "N/A"
                pos_elements = td[1].find_all('a', {'rel': 'nofollow'})
                player_data['Positions'] = ", ".join([span.text for pos in pos_elements for span in pos.find_all('span')] or ["N/A"])
                player_data['Age'] = td[2].text.strip() if td[2] else "N/A"
                player_data['Overall'] = td[3].text.strip() if td[3] else "N/A"
                player_data['Potential'] = td[4].text.strip() if td[4] else "N/A"
                player_data['Club'] = td[5].find('a').text.strip() if td[5].find('a') else "N/A"
                player_data['Height'] = td[6].text.strip() if td[6] else "N/A"
                player_data['Weight'] = td[7].text.strip() if td[7] else "N/A"
                player_data['Value'] = td[8].text.strip() if td[8] else "N/A"
                player_data['Wage'] = td[9].text.strip() if td[9] else "N/A"

                for i, col in enumerate(columns[12:], start=10):
                    player_data[col] = td[i].text.strip() if i < len(td) else "N/A"

                rv.append(pd.DataFrame([player_data], columns=columns))
            except Exception as e:
                print(f"Lỗi khi xử lý hàng tại offset {offset}: {e}")
                continue
        return pd.concat(rv, ignore_index=True) if rv else pd.DataFrame(columns=columns)
    except Exception as e:
        print(f"Lỗi khi gửi yêu cầu cho offset {offset}: {e}")
        return pd.DataFrame(columns=columns)

# Lặp qua các trang
for offset in range(0, 300):  # Lấy 2 trang để thử nghiệm
    print(f"Đang xử lý offset {offset}...", end="\r")
    page_data = get_data(offset)
    data = pd.concat([data, page_data], ignore_index=True)
    time.sleep(5)  # Chờ 5 giây để tránh bị chặn
    # Lưu tạm thời sau mỗi 10 trang
    if (offset + 1) % 10 == 0:
        data.to_csv(f"data_temp_offset_{offset + 1}.csv", index=False, encoding='utf-8-sig')
        print(f"Đã lưu tạm thời tại offset {offset + 1}")

# Loại bỏ trùng lặp và lưu file cuối
data = data.drop_duplicates()
print("\nDữ liệu đầu tiên:")
print(data.head())

# Lưu vào file CSV
data.to_csv("player_data.csv", index=False, encoding='utf-8-sig')
print("Đã lưu dữ liệu vào player_data.csv")

Đang truy cập URL: https://sofifa.com/players?showCol%5B0%5D=ae&showCol%5B1%5D=hi&showCol%5B2%5D=wi&showCol%5B3%5D=oa&showCol%5B4%5D=pt&showCol%5B5%5D=vl&showCol%5B6%5D=wg&showCol%5B7%5D=ta&showCol%5B8%5D=cr&showCol%5B9%5D=fi&showCol%5B10%5D=he&showCol%5B11%5D=sh&showCol%5B12%5D=vo&showCol%5B13%5D=ts&showCol%5B14%5D=dr&showCol%5B15%5D=cu&showCol%5B16%5D=fr&showCol%5B17%5D=lo&showCol%5B18%5D=bl&showCol%5B19%5D=to&showCol%5B20%5D=ac&showCol%5B21%5D=sp&showCol%5B22%5D=ag&showCol%5B23%5D=re&showCol%5B24%5D=ba&showCol%5B25%5D=tp&showCol%5B26%5D=so&showCol%5B27%5D=ju&showCol%5B28%5D=st&showCol%5B29%5D=sr&showCol%5B30%5D=ln&showCol%5B31%5D=te&showCol%5B32%5D=ar&showCol%5B33%5D=in&showCol%5B34%5D=po&showCol%5B35%5D=vi&showCol%5B36%5D=pe&showCol%5B37%5D=cm&showCol%5B38%5D=td&showCol%5B39%5D=ma&showCol%5B40%5D=sa&showCol%5B41%5D=sl&showCol%5B42%5D=tg&showCol%5B43%5D=gd&showCol%5B44%5D=gh&showCol%5B45%5D=gc&showCol%5B46%5D=gp&showCol%5B47%5D=gr&showCol%5B48%5D=tt&showCol%5B49%5D=pac&showCol%5B50%